In [1]:
import os;
print(os.getcwd())

/Users/ammulu/PycharmProjects/credit-risk-model


In [2]:
import os;

print(os.listdir('data/'))

['cs-training.csv']


In [3]:
import pandas as pd

file_path = 'data/cs-training'

try:
    df=pd.read_csv(file_path)
    print("Data loaded!")
    print(f"Dataset shape: {df.shape}")
    display(df.head())
except Exception as e:
    print(f"Error: {e}")

Error: [Errno 2] No such file or directory: 'data/cs-training'


In [6]:
import pandas as pd

df = pd.read_csv('data/cs-training.csv')

print(df.shape)
print(df.head())

(150000, 12)
   Id  SeriousDlqin2yrs RevolvingUtilizationOfUnsecuredLines  age  \
0   1                 1                          0.766126609   45   
1   2                 0                          0.957151019   40   
2   3                 0                           0.65818014   38   
3   4                 0                          0.233809776   30   
4   5                 0                            0.9072394   49   

   NumberOfTime30-59DaysPastDueNotWorse    DebtRatio  MonthlyIncome  \
0                                     2  0.802982129         9120.0   
1                                     0  0.121876201         2600.0   
2                                     1  0.085113375         3042.0   
3                                     0  0.036049682         3300.0   
4                                     1  0.024925695        63588.0   

   NumberOfOpenCreditLinesAndLoans  NumberOfTimes90DaysLate  \
0                               13                        0   
1                  

In [8]:
#We convert column to numbers.
df['RevolvingUtilizationOfUnsecuredLines'] = pd.to_numeric(df['RevolvingUtilizationOfUnsecuredLines'], errors='coerce')
df['DebtRatio'] = pd.to_numeric(df['DebtRatio'], errors='coerce')

# Fill missing Income with the middle value (Median)
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())

# Fill missing Dependents with 0
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)

# Fix the 'Age 0' error
df.loc[df['age'] < 18, 'age'] = df['age'].median()

# Fix the '98' error code in delinquency columns
late_cols = ['NumberOfTime30-59DaysPastDueNotWorse', 
             'NumberOfTime60-89DaysPastDueNotWorse', 
             'NumberOfTimes90DaysLate']
for col in late_cols:
    df.loc[df[col] >= 96, col] = df[col].median()

In [11]:
# Create a total score of how many times they've been late
df['TotalTimesLate'] = df['NumberOfTime30-59DaysPastDueNotWorse'] + \
                       df['NumberOfTime60-89DaysPastDueNotWorse'] + \
                       df['NumberOfTimes90DaysLate']

# Create a 'financial breathing room' column
df['IncomePerPerson'] = df['MonthlyIncome'] / (df['NumberOfDependents'] + 1)

In [13]:
df['TotalTimesLate'] = df['NumberOfTime30-59DaysPastDueNotWorse'] + df['NumberOfTimes90DaysLate'] + df['NumberOfTime60-89DaysPastDueNotWorse']
df['IncomePerPerson'] = df['MonthlyIncome'] / (df['NumberOfDependents'] + 1)

# The 'Id' column is just a serial number; it has no predictive power.
if 'Id' in df.columns:
    df = df.drop(columns=['Id'])

print("Preprocessing Complete! Your data is now 'Clean'.")
print(f"Current Data Shape: {df.shape}")
df.head()

Preprocessing Complete! Your data is now 'Clean'.
Current Data Shape: (150000, 13)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,TotalTimesLate,IncomePerPerson
0,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0,2,3040.0
1,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0,0,1300.0
2,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0,2,3042.0
3,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0,0,3300.0
4,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0,1,63588.0


In [8]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix

# 1. DEFINING INPUTS AND OUTPUTS
# 'y' is what we want to predict (Did they default?)
# 'X' is all the data we use to make that prediction
y = df['SeriousDlqin2yrs']
X = df.drop(columns=['SeriousDlqin2yrs'])

#2 DATA SPLITTING
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. INITIALIZING THE MODEL
# scale_pos_weight is a special setting for beginners to remember:
# Since we have very few 'Defaults', we tell the model to pay 12x 
# more attention to them so it doesn't get lazy.
model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    scale_pos_weight=12,
    random_state=42
)

# 4. START THE TRAINING (The Studying Phase)
print("Training the model... please wait a few seconds.")
model.fit(X_train, y_train)

# 5. THE FINAL EXAM (Predicting on data the model hasn't seen)
# We get the probability (a number between 0 and 1) for each person
probabilities = model.predict_proba(X_test)[:, 1]

# 6. CALCULATE THE SCORE
score = roc_auc_score(y_test, probabilities)

print("-" * 30)
print(f"FINAL SCORE (AUC-ROC): {score:.4f}")
print("-" * 30)
print("If your score is above 0.80, your model is performing well!")

Training the model... please wait a few seconds.
------------------------------
FINAL SCORE (AUC-ROC): 0.8692
------------------------------
If your score is above 0.80, your model is performing well!


In [9]:
!pip install xgboost

In [9]:
import mlflow
import mlflow.xgboost
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix

# 1. Start or set an experiment name in MLflow
mlflow.set_experiment("Credit_Risk_Model_Training")

# 2. DEFINING INPUTS AND OUTPUTS
y = df['SeriousDlqin2yrs']
X = df.drop(columns=['SeriousDlqin2yrs'])

# 3. DATA SPLITTING
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 4. START THE MLFLOW LOGGING RUN
with mlflow.start_run():

    # 5. INITIALIZING THE MODEL
    # We define the hyperparameters here
    n_estimators = 100
    learning_rate = 0.1
    max_depth = 5
    scale_pos_weight = 12

    model = XGBClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        scale_pos_weight=scale_pos_weight,
        random_state=42
    )

    # Log our hyperparameters to MLflow so we remember them
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("scale_pos_weight", scale_pos_weight)

    # 6. START THE TRAINING
    print("Training the model... please wait a few seconds.")
    model.fit(X_train, y_train)

    # 7. THE FINAL EXAM
    probabilities = model.predict_proba(X_test)[:, 1]

    # 8. CALCULATE AND LOG THE SCORE
    score = roc_auc_score(y_test, probabilities)

    # Automatically log AND register the model into the MLflow Model Registry!
    mlflow.xgboost.log_model(
        model,
        artifact_path="model",
        registered_model_name="Credit_Risk_XGB_Model"
    )

    # Log the performance metric to MLflow
    mlflow.log_metric("auc_roc", score)

    # Automatically log the trained XGBoost model structure itself!
    mlflow.xgboost.log_model(model, artifact_path="model")

    print("-" * 30)
    print(f"FINAL SCORE (AUC-ROC): {score:.4f}")
    print("-" * 30)
    print("MLflow successfully tracked this run!")

Training the model... please wait a few seconds.


2026/05/19 03:44:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'Credit_Risk_XGB_Model'.
Created version '1' of model 'Credit_Risk_XGB_Model'.
2026/05/19 03:44:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


------------------------------
FINAL SCORE (AUC-ROC): 0.8674
------------------------------
MLflow successfully tracked this run!


In [4]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Getting the importance scores from the model
importances = model.feature_importances_
feature_names = X.columns

# 2. Organizing them into a table (DataFrame)
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# 3. Create a Bar Chart
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='viridis')
plt.title('What Factors Drive Credit Risk? (Feature Importance)')
plt.xlabel('Importance Score')
plt.ylabel('Factors')
plt.show()

# 4. Print the raw numbers
print("Top 5 Most Important Factors:")
print(feature_importance_df.head(5))

NameError: name 'model' is not defined

In [12]:
import numpy as np  

# 1. Predicting '0' (Safe) or '1' (Default) for the test set
predictions = model.predict(X_test)

# 2. Counting how many of each using the numpy 'unique' function
unique, counts = np.unique(predictions, return_counts=True)
results = dict(zip(unique, counts))

print(f"In our test group of {len(predictions)} people:")

# We use .get(1, 0) to handle cases where there might be 0 rejects
approvals = results.get(0, 0)
rejects = results.get(1, 0)

print(f"- We would approve {approvals} loans.")
print(f"- We would reject {rejects} loans as high-risk.")

In our test group of 30000 people:
- We would approve 23746 loans.
- We would reject 6254 loans as high-risk.


In [13]:
import joblib

# This saves your trained model to your Macbook hard drive
joblib.dump(model, "credit_risk_model.joblib")
print("Model saved perfectly as credit_risk_model.joblib!")

Model saved perfectly as credit_risk_model.joblib!


In [14]:
import json
print(json.dumps(list(model.get_booster().feature_names)))

["RevolvingUtilizationOfUnsecuredLines", "age", "NumberOfTime30-59DaysPastDueNotWorse", "DebtRatio", "MonthlyIncome", "NumberOfOpenCreditLinesAndLoans", "NumberOfTimes90DaysLate", "NumberRealEstateLoansOrLines", "NumberOfTime60-89DaysPastDueNotWorse", "NumberOfDependents", "TotalTimesLate", "IncomePerPerson"]
